# FABRIC: attacker and victim testbed

Creates exactly two Ubuntu VMs at the site holding your existing persistent
block-storage allocation:

```text
existing attack-data volume --> attacker (192.168.100.1)
                                      |
                              private L2 experiment link
                                      |
                               victim (192.168.100.2)
                               generic / SKB XDP
```

Run this notebook in **FABRIC JupyterHub**, after completing FABRIC's
**Configure your Jupyter Environment** notebook (project, token, bastion key,
slice key and SSH configuration). The Python kernel needs
`fabrictestbed-extensions`; JupyterHub supplies it. Package installation on the
VMs requires their management network to reach Ubuntu's repositories.

Your project lead must already have allocated a persistent volume at `SITE`.
Set its exact name below. It must contain an existing, cleanly unmounted
filesystem and the attack data, and must be detached from any other VM.
This notebook attaches that allocation; it does not allocate a new volume,
format a disk, or download the dataset. Root disks are ephemeral.

The dataset is mounted **read-only** without journal replay. Management SSH
stays on FABRIC's management interface; experiment traffic uses a separate
Basic NIC on each VM. No attack replay is started by this notebook.

In [ ]:
from ipaddress import IPv4Address, IPv4Network
from pathlib import PurePosixPath
import json
import re
import shlex
import uuid

SLICE_NAME = "ebpf-ddos-testbed"
SITE = "REPLACE_WITH_STORAGE_SITE"
STORAGE_NAME = "REPLACE_WITH_EXISTING_VOLUME_NAME"
IMAGE = "default_ubuntu_24"
LEASE_HOURS = 24

# On a subsequent run, explicitly reconnect to this notebook's existing slice.
REUSE_SLICE = False

ATTACKER_RESOURCES = dict(cores=2, ram=8, disk=20)  # RAM/disk in GB
VICTIM_RESOURCES = dict(cores=2, ram=8, disk=20)
SUBNET = IPv4Network("192.168.100.0/24")
ATTACKER_IP = IPv4Address("192.168.100.1")
VICTIM_IP = IPv4Address("192.168.100.2")
DATA_MOUNT = "/mnt/attack_data"
DATA_SUBDIR = "DAT300-data"  # Relative directory on the volume, e.g. "CICDDoS2019"

# None: select the only filesystem on the attached volume. If there are several,
# inspect the printed lsblk output and enter the desired filesystem's UUID.
DATA_FS_UUID = None

if SITE.startswith("REPLACE_") or STORAGE_NAME.startswith("REPLACE_"):
    raise ValueError("Set SITE and STORAGE_NAME to your existing allocation.")
if ATTACKER_IP == VICTIM_IP or any(
    ip not in SUBNET or ip in (SUBNET.network_address, SUBNET.broadcast_address)
    for ip in (ATTACKER_IP, VICTIM_IP)
):
    raise ValueError("Choose distinct host addresses inside SUBNET.")
if not DATA_MOUNT.startswith("/mnt/") or re.search(r"\s", DATA_MOUNT):
    raise ValueError("DATA_MOUNT must be an absolute path under /mnt without whitespace.")
if ".." in PurePosixPath(DATA_MOUNT).parts:
    raise ValueError("DATA_MOUNT must not contain '..'.")
if PurePosixPath(DATA_SUBDIR).is_absolute() or ".." in PurePosixPath(DATA_SUBDIR).parts:
    raise ValueError("DATA_SUBDIR must stay inside the mounted volume.")

## Connect to FABRIC

Use the configured project containing the volume. Check site availability here
if provisioning fails; changing to an arbitrary site will not move the volume.

In [ ]:
from fabrictestbed_extensions.fablib.fablib import FablibManager

fablib = FablibManager()
fablib.list_sites()

## Provision or reconnect

The blocking submission waits for the VMs and FABlib's network configuration.
It can take several minutes. A same-name slice is never silently replaced.
To reconnect after a kernel restart, set `REUSE_SLICE = True` and retain the
original configuration. If submission times out, inspect the slice in the
FABRIC portal before resubmitting. Only a `StableOK` slice can be reused here.

In [ ]:
matches = [s for s in fablib.get_slices() if s.get_name() == SLICE_NAME]
if len(matches) > 1:
    raise RuntimeError("Multiple same-name slices exist; resolve them in the portal.")
if matches:
    if not REUSE_SLICE:
        raise RuntimeError("Slice already exists. Set REUSE_SLICE=True to reconnect.")
    testbed = matches[0]
    if str(testbed.get_state()) != "StableOK":
        raise RuntimeError(f"Slice is {testbed.get_state()}; inspect it in the portal.")
else:
    if REUSE_SLICE:
        raise RuntimeError("No existing slice found. Set REUSE_SLICE=False to create it.")
    testbed = fablib.new_slice(name=SLICE_NAME)
    attacker = testbed.add_node(
        name="attacker", site=SITE, image=IMAGE, **ATTACKER_RESOURCES
    )
    victim = testbed.add_node(
        name="victim", site=SITE, image=IMAGE, **VICTIM_RESOURCES
    )
    attacker.add_storage(name=STORAGE_NAME, auto_mount=False)
    attacker_iface = attacker.add_component(model="NIC_Basic", name="experiment").get_interfaces()[0]
    victim_iface = victim.add_component(model="NIC_Basic", name="experiment").get_interfaces()[0]
    testbed.add_l2network(
        name="experiment-lan", interfaces=[attacker_iface, victim_iface], subnet=SUBNET
    )
    attacker_iface.set_ip_addr(addr=ATTACKER_IP, mode="config")
    victim_iface.set_ip_addr(addr=VICTIM_IP, mode="config")
    testbed.submit(wait=True, wait_timeout=1800, lease_in_hours=LEASE_HOURS)

# Refresh handles after provisioning, including OS device names assigned by FABRIC.
attacker = testbed.get_node(name="attacker")
victim = testbed.get_node(name="victim")
if {n.get_name() for n in testbed.get_nodes()} != {"attacker", "victim"}:
    raise RuntimeError("Expected exactly the attacker and victim VMs.")
for node in (attacker, victim):
    if node.get_site() != SITE:
        raise RuntimeError(f"{node.get_name()} is not at configured SITE.")
attacker_iface = attacker.get_component(name="experiment").get_interfaces()[0]
victim_iface = victim.get_component(name="experiment").get_interfaces()[0]
for iface, expected_ip in ((attacker_iface, ATTACKER_IP), (victim_iface, VICTIM_IP)):
    if str(iface.get_ip_addr()) != str(expected_ip):
        raise RuntimeError("Existing slice address differs from this configuration.")
    iface.config()
ATTACKER_DEV = attacker_iface.get_device_name()
VICTIM_DEV = victim_iface.get_device_name()
STORAGE_DEV = attacker.get_storage(name=STORAGE_NAME).get_device_name()
if not ATTACKER_DEV or not VICTIM_DEV or not STORAGE_DEV:
    raise RuntimeError("FABRIC has not assigned the expected network/storage devices.")
testbed.show()
testbed.list_nodes()
print(f"Experiment interfaces: attacker={ATTACKER_DEV}, victim={VICTIM_DEV}")
print(f"Persistent volume device: {STORAGE_DEV}")

## Install experiment tools

The attacker receives `tcpreplay` and `tcpdump`. The victim receives Clang/LLVM,
libbpf headers, build tools and bpftool for its running kernel. If Ubuntu no
longer publishes the matching `linux-tools` package, use a supported FABRIC
Ubuntu image/kernel before continuing. The notebook does not change the kernel.

In [ ]:
def run_checked(node, script, timeout=1800):
    """FABlib returns stdout/stderr, so require a sentinel after shell success."""
    marker = "FABRIC_SUCCESS_" + uuid.uuid4().hex
    wrapped = "set -euo pipefail\n" + script + "\nprintf '\\n" + marker + "\\n'\n"
    stdout, stderr = node.execute(
        "bash -c " + shlex.quote(wrapped), quiet=True, timeout=timeout
    )
    if not stdout.rstrip().endswith(marker):
        raise RuntimeError(f"{node.get_name()} failed:\n{stdout}\n{stderr}")
    output = stdout.rstrip()[:-len(marker)].rstrip()
    if output:
        print(output)
    if stderr.strip():
        print(stderr)
    return output


def run_python(node, source, *args, root=False):
    command = ["python3", "-c", source, *map(str, args)]
    if root:
        command.insert(0, "sudo")
    return run_checked(node, shlex.join(command))


for node in (attacker, victim):
    run_checked(node, """
sudo apt-get update
sudo env DEBIAN_FRONTEND=noninteractive apt-get install -y \
    iproute2 iputils-ping ethtool tcpdump python3
""")
run_checked(attacker, "sudo env DEBIAN_FRONTEND=noninteractive apt-get install -y tcpreplay")
run_checked(victim, r"""
sudo env DEBIAN_FRONTEND=noninteractive apt-get install -y \
    clang llvm libbpf-dev libelf-dev build-essential pkg-config \
    linux-tools-common "linux-tools-$(uname -r)"
bpftool version
clang --version
uname -r
test -r /sys/kernel/btf/vmlinux
sudo mkdir -p /sys/fs/bpf
if ! mountpoint -q /sys/fs/bpf; then
    sudo mount -t bpf bpf /sys/fs/bpf
fi
test "$(findmnt -n -o FSTYPE --target /sys/fs/bpf)" = bpf
""")

## Mount the existing attack dataset

The device comes from the allocated FABRIC storage component, rather than an
assumed `/dev/vdb`. A whole-disk filesystem or a single formatted partition is
detected automatically. For multiple filesystems, set `DATA_FS_UUID` and rerun
this cell. An empty/raw disk is rejected. Supported filesystems are ext2/3/4,
XFS and Btrfs. The read-only mount is recorded by UUID in `/etc/fstab` so it
survives VM reboots. The mountpoint and dataset directory must contain data.

In [ ]:
mount_source = r"""
import json
from pathlib import Path
import subprocess
import sys

device, mountpoint, subdir, wanted_uuid = sys.argv[1:]

def output(*args):
    return subprocess.check_output(args, text=True).strip()

tree = json.loads(output("lsblk", "--json", "--paths", "--output",
                         "NAME,TYPE,FSTYPE,UUID,MOUNTPOINTS", device))
print(json.dumps(tree, indent=2))

def flatten(items):
    for item in items:
        yield item
        yield from flatten(item.get("children", []))

devices = list(flatten(tree["blockdevices"]))
supported = {"ext2": "ro", "ext3": "ro,noload", "ext4": "ro,noload",
             "xfs": "ro,norecovery", "btrfs": "ro,nologreplay"}
filesystems = [d for d in devices if d.get("fstype") in supported and d.get("uuid")]
if wanted_uuid:
    filesystems = [d for d in filesystems if d["uuid"] == wanted_uuid]
if len(filesystems) != 1:
    raise RuntimeError("Expected one supported filesystem on the allocated volume. "
                       "Check lsblk and set DATA_FS_UUID if needed. Never format existing data.")
fs = filesystems[0]
source = fs["name"]
options = supported[fs["fstype"]] + ",nofail"
target = Path(mountpoint)
target.mkdir(parents=True, exist_ok=True)

# Reject mounts of this volume elsewhere and conflicting mounts at our target.
for d in devices:
    for existing in d.get("mountpoints") or []:
        if existing and existing != mountpoint:
            raise RuntimeError(f"Volume already mounted at {existing}; inspect before proceeding.")
if target.is_mount():
    actual_uuid = output("findmnt", "-n", "-o", "UUID", "--mountpoint", mountpoint)
    actual_options = output("findmnt", "-n", "-o", "OPTIONS", "--mountpoint", mountpoint).split(",")
    if actual_uuid != fs["uuid"] or "ro" not in actual_options:
        raise RuntimeError("Mountpoint is occupied by a different or writable filesystem.")
else:
    if any(target.iterdir()):
        raise RuntimeError("Mountpoint contains local files; refusing to hide them.")
    subprocess.run(["mount", "-t", fs["fstype"], "-o", options, source, mountpoint], check=True)

data = (target / subdir).resolve()
if not data.is_relative_to(target.resolve()) or not data.is_dir():
    raise RuntimeError("Dataset directory must exist inside the volume.")
if not any(data.iterdir()):
    raise RuntimeError("Dataset directory is empty; check DATA_SUBDIR and the volume.")

# Reject competing fstab entries; rerunning this cell does not duplicate entries.
fstab = Path("/etc/fstab")
lines = fstab.read_text().splitlines()
entry = f"UUID={fs['uuid']} {mountpoint} {fs['fstype']} {options} 0 0"
kept = []
for line in lines:
    fields = line.split()
    if fields and not line.lstrip().startswith("#") and len(fields) >= 2:
        if fields[1] == mountpoint or fields[0] in (source, f"UUID={fs['uuid']}"):
            if fields[1] != mountpoint or fields[0] != f"UUID={fs['uuid']}":
                raise RuntimeError("Conflicting fstab entry; inspect /etc/fstab.")
            continue
    kept.append(line)
fstab.write_text("\n".join(kept + [entry]) + "\n")
print(output("findmnt", "--mountpoint", mountpoint))
print(f"Attack data directory: {data}")
for path in sorted(data.iterdir())[:10]:
    print(path.name)
"""
run_python(attacker, mount_source, STORAGE_DEV, DATA_MOUNT, DATA_SUBDIR,
           DATA_FS_UUID or "", root=True)

## Verify generic XDP on the victim

Compile a minimal `XDP_PASS` program, attach it with **`xdpgeneric`**, check
that the kernel reports **SKB mode (mode 2)**, ping through it, then detach it.
The test refuses to replace an existing XDP program. It runs only on the
victim's experiment interface and removes its own attachment even if ping fails.
The detector itself is not loaded by this test.

In [ ]:
probe_source = r"""
import json
from pathlib import Path
import subprocess
import sys
import tempfile

dev, peer = sys.argv[1:]

def link():
    return json.loads(subprocess.check_output(
        ["ip", "-j", "-d", "link", "show", "dev", dev], text=True))[0]

if link().get("xdp"):
    raise RuntimeError("An XDP program is already attached; inspect it before rerunning.")
with tempfile.TemporaryDirectory(prefix="fabric-xdp-") as work:
    source = Path(work) / "pass.c"
    obj = Path(work) / "pass.o"
    source.write_text('#include <linux/bpf.h>\n'
                      '#include <bpf/bpf_helpers.h>\n'
                      'SEC("xdp") int fabric_pass(struct xdp_md *ctx) { return XDP_PASS; }\n'
                      'char LICENSE[] SEC("license") = "GPL";\n')
    arch = subprocess.check_output(["gcc", "-dumpmachine"], text=True).strip()
    subprocess.run(["clang", "-O2", "-g", "-target", "bpf",
                    "-I/usr/include/" + arch, "-c", str(source), "-o", str(obj)], check=True)
    subprocess.run(["ip", "link", "set", "dev", dev, "xdpgeneric",
                    "obj", str(obj), "sec", "xdp"], check=True)
    try:
        status = link()
        print(json.dumps(status, indent=2))
        if status.get("xdp", {}).get("mode") != 2:
            raise RuntimeError("Expected generic XDP / SKB mode (2).")
        subprocess.run(["ping", "-I", dev, "-c", "3", "-W", "2", peer], check=True)
    finally:
        subprocess.run(["ip", "link", "set", "dev", dev, "xdpgeneric", "off"], check=True)
print("Generic XDP verified; temporary program detached.")
"""
run_python(victim, probe_source, VICTIM_DEV, ATTACKER_IP, root=True)
run_checked(attacker, shlex.join([
    "ping", "-I", ATTACKER_DEV, "-c", "3", "-W", "2", str(VICTIM_IP)
]))

## Connection details and next steps

Use the experiment interface names below for the detector and packet replay.
When replaying stored Ethernet captures, rewrite their destination MAC to the
victim's experiment MAC (and IPs as required by your experiment); the original
dataset addresses will not automatically match this topology.

After uploading this repository to the victim, its existing generic-XDP build
and attachment commands are:

```bash
cd eBPF-DDoS-Detection
clang -O2 -g -target bpf -I/usr/include/$(uname -m)-linux-gnu \
    -c kernelspace/heavykeeper/heavy_keeper.c -o heavy_keeper.o
sudo ip link set dev "$VICTIM_DEV" xdpgeneric obj heavy_keeper.o sec xdp
```

Replace `$VICTIM_DEV` with the printed interface name. The Go detector also
requires the Go version declared in `go.mod`; Go and the repository are not
installed here. After a reboot, rerun the provisioning cell with
`REUSE_SLICE=True` to restore/check interface configuration and rerun the
checks. XDP programs must be reattached after reboot.

In [ ]:
details = {
    "slice": SLICE_NAME,
    "site": SITE,
    "lease_end": str(testbed.get_lease_end()),
    "attacker": {
        "management_ip": str(attacker.get_management_ip()),
        "experiment_ip": str(ATTACKER_IP),
        "interface": ATTACKER_DEV,
        "mac": attacker_iface.get_mac(),
        "data_directory": str(PurePosixPath(DATA_MOUNT) / DATA_SUBDIR),
        "storage_volume": STORAGE_NAME,
    },
    "victim": {
        "management_ip": str(victim.get_management_ip()),
        "experiment_ip": str(VICTIM_IP),
        "interface": VICTIM_DEV,
        "mac": victim_iface.get_mac(),
        "xdp_mode": "generic (SKB)",
    },
}
print(json.dumps(details, indent=2))
for node in (attacker, victim):
    print(f"\n{node.get_name()} SSH: {node.get_ssh_command()}")

## Lease management and cleanup (manual)

VMs expire when their slice lease ends. Save victim results outside its root
disk before expiration or deletion. These commands are comments so **Run All**
keeps the testbed available. Persistent block-storage allocation and existing
data survive slice deletion; its original allocation expiry still applies.

In [ ]:
# Extend the lease when needed (subject to your project's lease limits):
# testbed.renew(days=1)

# Release the VMs and detach the existing persistent volume when finished:
# run_checked(attacker, "sudo umount " + shlex.quote(DATA_MOUNT))
# testbed.delete()

## References

- [Official FABRIC persistent-storage example](https://github.com/fabric-testbed/jupyter-examples/blob/main/fabric_examples/fablib_api/persistent_storage/persistent_storage.ipynb)
- [FABlib Node API: add_storage](https://fabric-fablib.readthedocs.io/en/latest/node.html)
- [FABlib Slice API](https://fabric-fablib.readthedocs.io/en/latest/slice.html)
- [FABlib Interface API](https://fabric-fablib.readthedocs.io/en/latest/interface.html)

The notebook intentionally mounts an already populated volume without the
formatting step shown in FABRIC's first-use storage example.